# Day 1 — Can we trust our data?

Before measuring anything about the A/B test, we need to understand what the
data contains, how reliable it is, and what has to be decided before any KPI
can be computed.

This notebook is an investigation, not a cleaning script. Every issue found is
documented, and the decision about how to handle it is written down next to the
evidence that motivated it.

## Setup

In [1]:
import pandas as pd
import numpy as np
from scipy import stats

from project_template.paths import RAW_DIR
from project_template.config import CONFIG

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

FILES = CONFIG["files"]
FUNNEL = CONFIG["funnel"]
FUNNEL

['start', 'step_1', 'step_2', 'step_3', 'confirm']

## 1. The three datasets

`df_final_demo` describes who the clients are, `df_final_experiment_clients`
records which arm of the experiment they were assigned to, and the two web
files are the raw interaction log, which has to be concatenated before use.

In [2]:
demo = pd.read_csv(RAW_DIR / FILES["demo"])
roster = pd.read_csv(RAW_DIR / FILES["experiment"])
web = pd.concat(
    [pd.read_csv(RAW_DIR / f) for f in FILES["web"]],
    ignore_index=True,
)

for name, df in [("demo", demo), ("roster", roster), ("web", web)]:
    print(f"{name:8} {df.shape[0]:>8,} rows x {df.shape[1]} columns")

demo       70,609 rows x 9 columns
roster     70,609 rows x 2 columns
web       755,405 rows x 5 columns


In [3]:
demo.head()

,client_id,clnt_tenure_yr,clnt_tenure_mnth,clnt_age,gendr,num_accts,bal,calls_6_mnth,logons_6_mnth
0,836976,6.0,73.0,60.5,U,2.0,45105.30,6.0,9.0
1,2304905,7.0,94.0,58.0,U,2.0,110860.30,6.0,9.0
2,1439522,5.0,64.0,32.0,U,2.0,52467.79,6.0,9.0
3,1562045,16.0,198.0,49.0,M,2.0,67454.65,3.0,6.0
4,5126305,12.0,145.0,33.0,F,2.0,103671.75,0.0,3.0


In [4]:
roster.head()

,client_id,Variation
0,9988021,Test
1,8320017,Test
2,4033851,Control
3,1982004,Test
4,9294070,Control


In [5]:
web.head()

,client_id,visitor_id,visit_id,process_step,date_time
0,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:27:07
1,9988021,580560515_7732621733,781255054_21935453173_531117,step_2,2017-04-17 15:26:51
2,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:19:22
3,9988021,580560515_7732621733,781255054_21935453173_531117,step_2,2017-04-17 15:19:13
4,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:18:04


### How the datasets relate

`client_id` is the key that links all three. Before merging anything, it is
worth checking how much they actually overlap: a client can appear in the
demographics file without ever having visited the site, and vice versa.

In [6]:
ids_demo = set(demo.client_id)
ids_roster = set(roster.client_id)
ids_web = set(web.client_id)

print(f"clients in demo        {len(ids_demo):>8,}")
print(f"clients in roster      {len(ids_roster):>8,}")
print(f"clients in web log     {len(ids_web):>8,}")
print()
print(f"demo == roster?        {ids_demo == ids_roster}")
print(f"in web but not demo    {len(ids_web - ids_demo):>8,}")
print(f"in demo but never web  {len(ids_demo - ids_web):>8,}")

clients in demo          70,609
clients in roster        70,609
clients in web log      120,157

demo == roster?        True
in web but not demo      49,548
in demo but never web         0


**Question to answer:** what does each dataset contribute, and which clients
are we able to analyse at all?

## 2. Data quality

Missing values, duplicates, data types and any value that does not make sense
for what the column is supposed to represent.

In [7]:
for name, df in [("demo", demo), ("roster", roster), ("web", web)]:
    missing = df.isna().sum()
    missing = missing[missing > 0]
    print(f"--- {name} ---")
    print(missing.to_string() if len(missing) else "no missing values")
    print()

--- demo ---
clnt_tenure_yr      14
clnt_tenure_mnth    14
clnt_age            15
gendr               14
num_accts           14
bal                 14
calls_6_mnth        14
logons_6_mnth       14

--- roster ---
Variation    20109

--- web ---
no missing values



The missing values in `demo` affect almost every column at once, which suggests
whole rows are empty rather than individual fields being unrecorded. Worth
looking at directly.

In [8]:
incomplete = demo[demo.isna().any(axis=1)]
print(f"{len(incomplete)} rows with at least one missing value")
incomplete

15 rows with at least one missing value


,client_id,clnt_tenure_yr,clnt_tenure_mnth,clnt_age,gendr,num_accts,bal,calls_6_mnth,logons_6_mnth
4164,7402828,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8316,355337,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8677,8412164,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9583,4666211,8.0,106.0,NaN,F,2.0,42550.55,4.0,7.0
13444,2222915,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
18066,4876926,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25961,5277910,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
28432,7616759,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
35323,8191345,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
43518,1227228,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


`Variation` is missing for a large number of clients. This is not a data entry
problem: it identifies clients who were never part of the experiment.

In [9]:
roster.Variation.value_counts(dropna=False)

Variation
Test       26968
Control    23532
NaN        20109
Name: count, dtype: int64

### Duplicates

In [10]:
for name, df in [("demo", demo), ("roster", roster), ("web", web)]:
    print(f"{name:8} {df.duplicated().sum():>8,} fully duplicated rows")

demo            0 fully duplicated rows
roster          0 fully duplicated rows


web        10,764 fully duplicated rows


Exact duplicates in an event log are ambiguous. They could be a tracking
artefact firing the same event twice, or a genuine repeated interaction that
happens to share a timestamp. Look at examples before deciding.

In [11]:
dupes = web[web.duplicated(keep=False)].sort_values(["visit_id", "date_time"])
print(f"{len(dupes):,} rows involved in a duplicate")
dupes.head(12)

19,816 rows involved in a duplicate


,client_id,visitor_id,visit_id,process_step,date_time
501226,3852267,662624447_15868585035,10024125_38177552152_999695,start,2017-05-01 09:03:30
501227,3852267,662624447_15868585035,10024125_38177552152_999695,start,2017-05-01 09:03:30
618525,5269746,241531822_62124427563,10043239_12589390657_342180,start,2017-05-01 19:45:14
618526,5269746,241531822_62124427563,10043239_12589390657_342180,start,2017-05-01 19:45:14
521125,4389198,729937373_63248482072,100618014_41463020246_787063,start,2017-05-11 09:13:46
521126,4389198,729937373_63248482072,100618014_41463020246_787063,start,2017-05-11 09:13:46
674698,916369,558452417_7504499072,100787296_75463077257_149957,start,2017-05-03 16:27:24
674699,916369,558452417_7504499072,100787296_75463077257_149957,start,2017-05-03 16:27:24
379902,8342684,220532119_57925398454,100810638_79925673386_145828,start,2017-05-11 02:41:07
379903,8342684,220532119_57925398454,100810638_79925673386_145828,start,2017-05-11 02:41:07


### Data types

In [12]:
web.dtypes

client_id       int64
visitor_id        str
visit_id          str
process_step      str
date_time         str
dtype: object

`date_time` arrives as text, so it cannot be sorted or subtracted until it is
parsed. Everything downstream — ordering events, measuring durations — depends
on this.

In [13]:
web["date_time"] = pd.to_datetime(web["date_time"])

print("parsed:", web.date_time.dtype)
print("failed to parse:", web.date_time.isna().sum())
print("range:", web.date_time.min(), "->", web.date_time.max())

parsed: datetime64[us]
failed to parse: 0
range: 2017-03-15 00:03:03 -> 2017-06-20 23:59:57


The experiment ran from 15 March to 20 June 2017. Comparing that against the
range above tells us whether the log is restricted to the experiment window or
contains activity from outside it.

In [14]:
start, end = CONFIG["experiment"]["start"], CONFIG["experiment"]["end"]
print("configured window:", start, "->", end)

outside = web[
    (web.date_time.dt.date < start) | (web.date_time.dt.date > end)
]
print(f"events outside the window: {len(outside):,}")

configured window: 2017-03-15 -> 2017-06-20


events outside the window: 0


### Do the values make sense?

In [15]:
demo[["clnt_age", "clnt_tenure_yr", "clnt_tenure_mnth",
      "bal", "num_accts", "calls_6_mnth", "logons_6_mnth"]].describe().T

,count,mean,std,min,25%,50%,75%,max
clnt_age,70594.0,46.442240,15.591273,13.50,32.500,47.0,59.000,96.00
clnt_tenure_yr,70595.0,12.052950,6.871819,2.00,6.000,11.0,16.000,62.00
clnt_tenure_mnth,70595.0,150.659367,82.089854,33.00,82.000,136.0,192.000,749.00
bal,70595.0,147445.240641,301508.706531,13789.42,37346.835,63332.9,137544.905,16320040.15
num_accts,70595.0,2.255528,0.534997,1.00,2.000,2.0,2.000,8.00
calls_6_mnth,70595.0,3.382478,2.236580,0.00,1.000,3.0,6.000,7.00
logons_6_mnth,70595.0,5.566740,2.353286,1.00,4.000,5.0,7.000,9.00


In [16]:
demo.gendr.value_counts(dropna=False)

gendr
U      24122
M      23724
F      22746
NaN       14
X          3
Name: count, dtype: int64

**Question to answer:** are there data quality issues that could affect the
analysis? Note anything unexpected in the distributions above — implausible
extremes, placeholder-looking values, or categories that need interpreting
rather than cleaning.

## 3. The identifiers

The expected hierarchy is:

```
client_id  ->  visitor_id  (browser / device)  ->  visit_id  (session)
```

A client can use several devices, and each device can produce several sessions.
Whether the data actually respects this determines which identifier we can use
as the unit of analysis for each question.

In [17]:
print(f"unique client_id   {web.client_id.nunique():>9,}")
print(f"unique visitor_id  {web.visitor_id.nunique():>9,}")
print(f"unique visit_id    {web.visit_id.nunique():>9,}")
print(f"total events       {len(web):>9,}")

unique client_id     120,157
unique visitor_id    130,236
unique visit_id      158,095
total events         755,405


If the hierarchy holds, every `visitor_id` belongs to exactly one client, and
every `visit_id` belongs to exactly one visitor.

In [18]:
visitor_to_client = web.groupby("visitor_id").client_id.nunique()
visit_to_visitor = web.groupby("visit_id").visitor_id.nunique()

print(f"visitor_id linked to >1 client_id:  {(visitor_to_client > 1).sum():>6,}")
print(f"visit_id linked to >1 visitor_id:   {(visit_to_visitor > 1).sum():>6,}")

visitor_id linked to >1 client_id:   1,645
visit_id linked to >1 visitor_id:        0


In [19]:
# How much activity does a single client generate?
per_client = web.groupby("client_id").agg(
    visits=("visit_id", "nunique"),
    devices=("visitor_id", "nunique"),
    events=("process_step", "size"),
)
per_client.describe().T

,count,mean,std,min,25%,50%,75%,max
visits,120157.0,1.324201,0.741575,1.0,1.0,1.0,1.0,21.0
devices,120157.0,1.097722,0.374796,1.0,1.0,1.0,1.0,14.0
events,120157.0,6.286816,3.986973,1.0,5.0,5.0,7.0,111.0


**Question to answer:** which identifier belongs at which level of the
analysis? The brief fixes `client_id` as the experimental unit, but journeys
are reconstructed from sessions.

## 4. Is the experiment itself trustworthy?

Before comparing outcomes, the assignment to Control and Test has to be
complete, unique and consistent.

In [20]:
print("rows in roster:", f"{len(roster):,}")
print("unique clients:", f"{roster.client_id.nunique():,}")
print("clients listed more than once:",
      (roster.client_id.value_counts() > 1).sum())

rows in roster: 70,609
unique clients: 70,609
clients listed more than once: 0


A client appearing in both arms would invalidate any comparison, so it is worth
checking explicitly rather than assuming.

In [21]:
assigned = roster.dropna(subset=["Variation"])
groups_per_client = assigned.groupby("client_id").Variation.nunique()

print("clients assigned to both groups:", (groups_per_client > 1).sum())

clients assigned to both groups: 0


### Is the split balanced?

Randomised assignment does not produce exactly equal groups, but it should not
stray far from the intended ratio either. A chi-square test against a 50/50
split shows whether the imbalance is larger than chance would explain.

In [22]:
counts = assigned.Variation.value_counts()
n_test, n_control = counts["Test"], counts["Control"]
total = n_test + n_control

print(f"Test     {n_test:>7,}   {n_test/total:.1%}")
print(f"Control  {n_control:>7,}   {n_control/total:.1%}")
print(f"total    {total:>7,}")
print()

result = stats.chisquare([n_test, n_control])
print(f"chi-square vs 50/50:  statistic = {result.statistic:.1f}   p = {result.pvalue:.3g}")

Test      26,968   53.4%
Control   23,532   46.6%
total     50,500

chi-square vs 50/50:  statistic = 233.8   p = 8.92e-53


The split is further from 50/50 than chance explains. That on its own does not
invalidate the experiment, but it does raise the question of **whether the
imbalance is related to who the clients are**. If one arm ended up with
systematically different clients, later comparisons would be confounded.

In [23]:
profiled = roster.merge(demo, on="client_id").dropna(subset=["clnt_age"])
profiled["age_band"] = pd.cut(
    profiled.clnt_age, [0, 35, 50, 65, 120],
    labels=["<35", "35-50", "50-65", "65+"],
)

in_experiment = profiled.dropna(subset=["Variation"])
by_age = pd.crosstab(in_experiment.age_band, in_experiment.Variation)
by_age["% Test"] = (by_age.Test / (by_age.Test + by_age.Control) * 100).round(2)

chi = stats.chi2_contingency(
    pd.crosstab(in_experiment.age_band, in_experiment.Variation)
)
print(f"age band vs group:  chi2 = {chi.statistic:.2f}   p = {chi.pvalue:.4f}")
print()
by_age

age band vs group:  chi2 = 5.99   p = 0.1122



Variation,Control,Test,% Test
age_band,,,
<35,6571,7757,54.14
35-50,6060,6968,53.48
50-65,7582,8566,53.05
65+,3313,3670,52.56


### Who are the unassigned clients?

The clients with no `Variation` were never randomised, so they cannot take part
in the comparison. Whether they resemble the rest of the client base determines
how much of a limitation that is.

In [24]:
profiled["in_experiment"] = profiled.Variation.notna()

summary = profiled.groupby("in_experiment").clnt_age.agg(
    ["count", "median", "mean"]
).round(1)

u, p = stats.mannwhitneyu(
    profiled[profiled.in_experiment].clnt_age,
    profiled[~profiled.in_experiment].clnt_age,
)
print(f"Mann-Whitney on age:  p = {p:.3g}")
print()
summary

Mann-Whitney on age:  p = 4.09e-119



,count,median,mean
in_experiment,,,
False,20107,45.0,44.2
True,50487,48.0,47.3


**Question to answer:** can every client be confidently assigned to a single
experimental group, and is the assignment itself sound?

## 5. The event log

The logs record events, not journeys. Understanding how those events behave is
what makes it possible to reconstruct journeys tomorrow.

In [25]:
step_counts = web.process_step.value_counts().reindex(FUNNEL)
step_counts.to_frame("events").assign(
    share=lambda d: (d.events / d.events.sum()).map("{:.1%}".format)
)

,events,share
process_step,,
start,243945,32.3%
step_1,163193,21.6%
step_2,133062,17.6%
step_3,112242,14.9%
confirm,102963,13.6%


### How long is a session?

In [26]:
events_per_visit = web.groupby("visit_id").size()
events_per_visit.describe().T

count    158095.000000
mean          4.778171
std           2.988947
min           1.000000
25%           2.000000
50%           5.000000
75%           6.000000
max         104.000000
dtype: float64

In [27]:
events_per_visit.value_counts().sort_index().head(15).to_frame("visits")

,visits
1,22484
2,18289
3,9877
4,11246
5,51649
6,12595
7,12795
8,5576
9,4680
10,2696


### The analytical challenges

Each of these has to be interpreted before any KPI is computed. Some are
genuine client behaviour, others are tracking problems. Telling them apart is
the point of this section.

In [28]:
per_visit = web.groupby("visit_id").process_step.value_counts().unstack(fill_value=0)

multi_start = (per_visit.get("start", 0) > 1).sum()
multi_confirm = (per_visit.get("confirm", 0) > 1).sum()
no_start = (per_visit.get("start", 0) == 0).sum()

print(f"visits with more than one start:    {multi_start:>8,}")
print(f"visits with more than one confirm:  {multi_confirm:>8,}")
print(f"visits with no start at all:        {no_start:>8,}")
print(f"total visits:                       {len(per_visit):>8,}")

visits with more than one start:      55,115
visits with more than one confirm:     8,965
visits with no start at all:          13,193
total visits:                        158,095


### Repeated steps and backward navigation

Mapping each step to its position in the funnel makes it possible to tell
forward progress from going back.

In [29]:
STEP_ORDER = {step: i for i, step in enumerate(FUNNEL)}

web = web.sort_values(["visit_id", "date_time"])
web["step_rank"] = web.process_step.map(STEP_ORDER)
web["prev_rank"] = web.groupby("visit_id").step_rank.shift()

web["is_repeat"] = web.step_rank == web.prev_rank
web["is_backward"] = web.step_rank < web.prev_rank

print(f"repeated steps:        {web.is_repeat.sum():>9,} events")
print(f"backward movements:    {web.is_backward.sum():>9,} events")
print()
print(f"visits with a repeat:   {web.groupby('visit_id').is_repeat.any().sum():>8,}")
print(f"visits going backwards: {web.groupby('visit_id').is_backward.any().sum():>8,}")

repeated steps:           87,980 events
backward movements:       63,878 events

visits with a repeat:     49,503


visits going backwards:   40,516


In [30]:
# How far back do people jump?
back = web[web.is_backward]
(back.prev_rank - back.step_rank).value_counts().sort_index().to_frame("events")

,events
1.0,45495
2.0,7594
3.0,8845
4.0,1944


### Exceptionally long sessions

A session spanning days is unlikely to be a slow completion. It is more likely
someone abandoned the process and returned later, with the tracker treating it
as one session.

In [31]:
duration = web.groupby("visit_id").date_time.agg(["min", "max"])
duration["minutes"] = (duration["max"] - duration["min"]).dt.total_seconds() / 60

print(duration.minutes.describe().to_string())
print()
for q in [0.90, 0.95, 0.99, 0.999]:
    print(f"p{q*100:<6} {duration.minutes.quantile(q):>12,.1f} min")
print()
print(f"sessions longer than 24h: {(duration.minutes > 1440).sum():,}")

count    158095.000000
mean          5.294590
std           9.943423
min           0.000000
25%           0.700000
50%           2.850000
75%           6.033333
max         712.800000

p90.0           12.0 min
p95.0           18.4 min
p99.0           41.8 min
p99.9          114.9 min

sessions longer than 24h: 0


### An example journey

Reading a few real visits end to end is worth more than any summary statistic
for understanding what the data actually looks like.

In [32]:
sample_visit = (
    web[web.is_backward].visit_id.iloc[0]
    if web.is_backward.any()
    else web.visit_id.iloc[0]
)

web[web.visit_id == sample_visit][
    ["client_id", "visit_id", "process_step", "date_time"]
].sort_values("date_time")

,client_id,visit_id,process_step,date_time
240562,7338123,100019538_17884295066_43909,start,2017-04-09 16:20:56
240561,7338123,100019538_17884295066_43909,step_1,2017-04-09 16:21:12
240560,7338123,100019538_17884295066_43909,step_2,2017-04-09 16:21:21
240559,7338123,100019538_17884295066_43909,step_1,2017-04-09 16:21:35
240558,7338123,100019538_17884295066_43909,step_1,2017-04-09 16:21:41
240557,7338123,100019538_17884295066_43909,start,2017-04-09 16:21:45
240556,7338123,100019538_17884295066_43909,start,2017-04-09 16:21:59
240555,7338123,100019538_17884295066_43909,step_1,2017-04-09 16:22:04
240554,7338123,100019538_17884295066_43909,step_2,2017-04-09 16:22:08
240553,7338123,100019538_17884295066_43909,step_3,2017-04-09 16:24:01


## Decisions taken

Everything above is evidence. This section turns it into a methodology, and it
is what the rest of the project relies on.

| Issue | What we observed | Decision | Rationale |
|---|---|---|---|
| Clients with no `Variation` | 20,109 clients, and they are younger than the assigned ones (median 45 vs 48, p ≈ 4e-119) | Excluded from every Control/Test comparison. Kept when describing the client base | They were never randomised, so including them would break the comparison. The age difference is recorded as a limitation on how far our conclusions generalise |
| Clients in the web log with no profile | 49,548 clients appear in the log but not in `demo` or the roster | Excluded from the A/B analysis and declared as a limitation | They cannot be assigned to a group or profiled, so no comparison is possible for them |
| Rows with missing demographics | 14–15 rows, empty across almost every column at once | Dropped | Empty records rather than unreported values, and 0.02% of the file |
| Exact duplicate events | 10,764 events identical in client, visit, step and timestamp | Dropped | The same interaction cannot occur twice in the same second: this is the tracker firing twice |
| `gendr` value `U` | 24,122 clients (34%) | Kept as its own category, read as "not reported" | It is a category, not a missing value. Dropping a third of the sample would bias every demographic comparison |
| Assignment imbalance | 53.4% Test vs 46.6% Control, chi-square p ≈ 9e-53 | Documented as a threat to validity, not corrected. Verified that it is **not** related to age (p = 0.11) | The imbalance is real and cannot be undone, but since the share of Test is flat across age bands, it does not confound comparisons by client profile |
| Multiple `start` per visit | 55,115 visits (35%) | Keep the first `start` of each visit; treat the visit as one attempt | Simpler to defend and it avoids inflating the denominator. Recorded as a methodological choice to revisit on day 3 |
| Multiple `confirm` per visit | 8,965 visits | Keep the first `confirm` | Confirming twice is not completing twice |
| Repeated steps | 87,980 events across 49,503 visits | Kept. Not counted as errors yet | Real behaviour, and the raw figure is inflated by the exact duplicates above. Has to be recomputed after deduplication |
| Backward navigation | 63,878 events across 40,516 visits | Kept. Whether it counts as an error is a day 3 decision | Going back is not necessarily confusion, and treating it as an error is a choice that changes the conclusion |
| Very long sessions | Median 2.9 min, p99 41.8 min, maximum 713 min. **No session exceeds 24 h** | No trimming applied | The brief warns about extreme durations, but they do not occur here. Reviewed and found unnecessary rather than ignored |
| Single-event visits | 25,022 visits with one event and zero duration | Kept in the denominator | Leaving immediately is a real outcome, not missing data |

**Final question:** are we confident the data is ready for KPI computation?

## Where this leaves us

**The data is usable, with two documented limitations.** Of the 70,609 clients
with a profile, 50,500 were randomised and can take part in the comparison. The
49,548 log-only clients and the 20,109 unassigned ones are out, and both
exclusions are recorded above.

**The experiment survives its main threat.** The 53/47 imbalance is severe
enough to be worth reporting, but the share of Test clients is essentially flat
across age bands (54.1%, 53.5%, 53.1%, 52.6%; p = 0.11). Any difference we
later find between client profiles cannot be attributed to a skewed
assignment — which is the first objection this analysis would otherwise face.

**The event log is dirtier than the funnel counts suggest.** A third of visits
contain more than one `start`, 8,965 contain more than one `confirm`, and
13,193 contain no `start` at all. Journeys cannot simply be read off the log:
they have to be reconstructed under rules stated in advance, which is the work
for day 3.

### What to pursue next

Two questions follow directly from today and shape the rest of the project.

**Does the redesign work equally well for everyone?** The client base is not
homogeneous, and the unassigned clients being measurably younger is a first
sign that age separates behaviour here. Comparing outcomes only on the average
would hide that.

**Is backward navigation a problem or a feature?** This is the decision that
matters most for day 3. The brief asks it directly, and the answer determines
what Error Rate measures. If backward movement counts as an error, a design
that lets people review their choices will score worse than one that does not —
even if more clients finish with it. The KPI definition, not the data, will
decide that conclusion, so it has to be argued rather than assumed.